# **Aprendizaje por refuerzos** - FrozenLake y LunarLander (Gymnasium)

## Tarea: Implementar Agentes Q-Learning y DQN

### Objetivos:
1. Implementar el algoritmo Q-Learning
2. Implementar el algoritmo DQN
3. Entrenar y evaluar ambos agentes
4. Comparar el rendimiento de ambos enfoques


In [1]:
# Instalar paquetes requeridos
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

%pip install swig matplotlib gymnasium torch pygame


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importar las bibliotecas
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import pygame
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import deque, namedtuple
import random


c:\Users\feder\Facultad\ApAut\Laboratorios\AA25\.venvAA25\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


La siguiente celda permite ejecutar un juego de Frozen Lake *determinista* para jugar con el teclado.

Utilize las teclas de dirección (flechas) o asdw para comandar al agente.


In [3]:
def jugar_frozen_lake(env):
    env.reset()
    
    print("Controles:")
    print("W - Arriba")
    print("S - Abajo") 
    print("A - Izquierda")
    print("D - Derecha")
    print("Q - Salir")
    print("Presione cualquier tecla para empezar...")
    
    pygame.init()
    pygame.display.set_caption("FrozenLake - Juego Interactivo")
    
    clock = pygame.time.Clock()
    ejecutando = True
    
    while ejecutando:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                ejecutando = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_q or event.key == pygame.K_ESCAPE:
                    ejecutando = False
                elif event.key == pygame.K_w or event.key == pygame.K_UP:
                    accion = 3  # Arriba
                elif event.key == pygame.K_s or event.key == pygame.K_DOWN:
                    accion = 1  # Abajo
                elif event.key == pygame.K_a or event.key == pygame.K_LEFT:
                    accion = 0  # Izquierda
                elif event.key == pygame.K_d or event.key == pygame.K_RIGHT:
                    accion = 2  # Derecha
                else:
                    continue
                
                observacion, recompensa, terminado, truncado, info = env.step(accion)
                print(f"Acción: {accion}, Recompensa: {recompensa}, Terminado: {terminado}")
                
                if terminado or truncado:
                    print(f"¡Episodio terminado! Recompensa final: {recompensa}")
                    pygame.time.wait(500)
                    env.reset()
        
        clock.tick(60)
    
    pygame.quit()
    env.close()

env = gym.make('FrozenLake-v1', render_mode='human', is_slippery=False)
# Descomente la línea de abajo para jugar interactivamente
# jugar_frozen_lake(env)


La siguiente celda permite jugar al juego no determinista.

In [4]:
env = gym.make('FrozenLake-v1', render_mode='human', is_slippery=True)
# Descomente la línea de abajo para jugar interactivamente
jugar_frozen_lake(env)

Controles:
W - Arriba
S - Abajo
A - Izquierda
D - Derecha
Q - Salir
Presione cualquier tecla para empezar...


La siguiente clase define la interfaz de los agentes que utilizaremos para jugar al Frozen Lake.


In [5]:
from abc import ABC, abstractmethod

class Agente(ABC):
    
    @abstractmethod
    def elegir_accion(self, estado):
        """Elige una acción dada una observación."""
        pass
    
    @abstractmethod
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Aprende de la experiencia."""
        pass

class AgenteAleatorio(Agente):
    """Agente aleatorio que elige acciones al azar."""
    
    def __init__(self, espacio_acciones):
        # Se guarda el espacio de acciones para poder elegir acciones al azar
        self.espacio_acciones = espacio_acciones
    
    def elegir_accion(self, estado):
        return self.espacio_acciones.sample()
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        pass  # El agente aleatorio no aprende

# Probar el AgenteAleatorio
env = gym.make('FrozenLake-v1')
agente_aleatorio = AgenteAleatorio(env.action_space)
estado, _ = env.reset()
accion = agente_aleatorio.elegir_accion(estado)
print(f"✓ AgenteAleatorio creado y probado. Acción: {accion}")
env.close()


✓ AgenteAleatorio creado y probado. Acción: 3


La siguiente celda define una función para evaluar el desempeño de un agente dado.

In [6]:
# Función de Evaluación de Agentes
def evaluar_agente(agente, env, num_episodios=1000):
    """
    Evalúa el rendimiento de un agente a lo largo de múltiples episodios.
    
    Args:
        agente: El agente a evaluar
        env: El entorno
        num_episodios: Número de episodios a ejecutar
    
    Returns:
        dict: Resultados de la evaluación
    """
    recompensas_totales = []
    victorias = 0
    
    for episodio in range(num_episodios):
        estado, _ = env.reset()
        recompensa_total = 0
        
        while True:
            accion = agente.elegir_accion(estado)
            estado, recompensa, terminado, truncado, _ = env.step(accion)
            recompensa_total += recompensa
            
            if terminado or truncado:
                break
        
        recompensas_totales.append(recompensa_total)
        if recompensa_total > 0:
            victorias += 1
    
    return {
        'recompensas_totales': recompensas_totales,
        'victorias': victorias,
        'tasa_victorias': victorias / num_episodios,
        'recompensa_promedio': np.mean(recompensas_totales),
        'desv_estandar': np.std(recompensas_totales)
    }

def imprimir_resultados_evaluacion(resultados, nombre_agente):
    """Imprime los resultados de evaluación de forma formateada."""
    print(f"\n{nombre_agente} - Resultados de Evaluación:")
    print(f"Tasa de Victorias: {resultados['tasa_victorias']:.1%}")
    print(f"Recompensa Promedio: {resultados['recompensa_promedio']:.3f}")
    print(f"Desviación Estándar: {resultados['desv_estandar']:.3f}")
    print(f"Total de Victorias: {resultados['victorias']}")

# Probar función de evaluación
env = gym.make('FrozenLake-v1')
agente_aleatorio = AgenteAleatorio(env.action_space)
resultados = evaluar_agente(agente_aleatorio, env, num_episodios=100)
imprimir_resultados_evaluacion(resultados, "Agente Aleatorio")
env.close()



Agente Aleatorio - Resultados de Evaluación:
Tasa de Victorias: 0.0%
Recompensa Promedio: 0.000
Desviación Estándar: 0.000
Total de Victorias: 0


La siguiente celda define una función para entrenar un agente.

In [ ]:
# Función de Entrenamiento de Agentes
def entrenar_agente(agente, env, num_episodios=1000, max_pasos=100, verbose=True):
    """
    Entrena un agente en el entorno.
    
    Args:
        agente: El agente a entrenar
        env: El entorno
        num_episodios: Número de episodios de entrenamiento
        max_pasos: Máximo de pasos por episodio
        verbose: Si imprimir el progreso
    
    Returns:
        list: Recompensas de episodios
    """
    recompensas_episodios = []
    longitudes_episodios = []
    num_episodios_10 = int(num_episodios / 10)
    
    for episodio in range(num_episodios):
        estado, _ = env.reset()
        recompensa_total = 0
        pasos = 0
        
        for paso in range(max_pasos):
            accion = agente.elegir_accion(estado)
            siguiente_estado, recompensa, terminado, truncado, _ = env.step(accion)
            
            agente.aprender(estado, accion, recompensa, siguiente_estado, terminado or truncado)
            
            estado = siguiente_estado
            recompensa_total += recompensa
            pasos += 1
            
            if terminado or truncado:
                break
        
        recompensas_episodios.append(recompensa_total)
        longitudes_episodios.append(pasos)
        
        if verbose and (episodio + 1) % num_episodios_10 == 0:
            recompensa_promedio = np.mean(recompensas_episodios[-num_episodios_10:])
            longitud_promedio = np.mean(longitudes_episodios[-num_episodios_10:])
            print(f"Episodio {episodio + 1}: Recompensa Promedio = {recompensa_promedio:.3f}, Longitud Promedio = {longitud_promedio:.1f}")
    
    return recompensas_episodios, longitudes_episodios

print("✓ Funciones de entrenamiento definidas")


✓ Funciones de entrenamiento definidas


In [ ]:
# Ejecución del Agente Aleatorio
env = gym.make('FrozenLake-v1')
agente_aleatorio = AgenteAleatorio(env.action_space)
entrenar_agente(agente_aleatorio, env)
resultados = evaluar_agente(agente_aleatorio, env, num_episodios=100)
imprimir_resultados_evaluacion(resultados, "Agente Aleatorio")
env.close()

Episodio 500: Recompensa Promedio = 0.010, Longitud Promedio = 7.5
Episodio 1000: Recompensa Promedio = 0.006, Longitud Promedio = 7.4
Episodio 1500: Recompensa Promedio = 0.016, Longitud Promedio = 7.7
Episodio 2000: Recompensa Promedio = 0.010, Longitud Promedio = 7.6
Episodio 2500: Recompensa Promedio = 0.010, Longitud Promedio = 7.8
Episodio 3000: Recompensa Promedio = 0.016, Longitud Promedio = 7.8
Episodio 3500: Recompensa Promedio = 0.020, Longitud Promedio = 7.4
Episodio 4000: Recompensa Promedio = 0.018, Longitud Promedio = 7.8
Episodio 4500: Recompensa Promedio = 0.018, Longitud Promedio = 7.8
Episodio 5000: Recompensa Promedio = 0.004, Longitud Promedio = 7.8

Agente Aleatorio - Resultados de Evaluación:
Tasa de Victorias: 1.0%
Recompensa Promedio: 0.010
Desviación Estándar: 0.099
Total de Victorias: 1


La siguiente celda define el agente de Q-Learning a implementar.

In [165]:
# TODO: Implementar Agente Q-Learning
class AgenteQLearning(Agente):
    """Agente que usa el algoritmo Q-Learning."""
    q_table = None
    q_visitas = None

    def __init__(self, action_space, cant_estados=16):
        self.q_table = np.zeros((cant_estados, 4), dtype=np.float64)
        self.q_visitas = np.zeros((cant_estados, 4), dtype=np.int32)
        self.action_space = action_space.n
        self.epsilon = 1  # Epsilon inicial alto
         # Configuración de epsilon según el tamaño del estado
        if cant_estados == 16:
            self.epsilon_decay = 0.999  # Decaimiento de epsilon
            self.epsilon_min = 0.01  # Epsilon mínimo
            self.gamma = 0.99
        elif cant_estados == 64:
            self.epsilon_decay = 0.9995  # Decaimiento de epsilon
            self.epsilon_min = 0.001  # Epsilon mínimo
            self.gamma = 0.99

    def elegir_accion(self, estado):
        """Elige una acción usando política epsilon-greedy."""
        if np.random.rand() < self.epsilon:
            return np.random.choice(self.action_space)  # Exploración
        else:
            return np.argmax(self.q_table[estado])
        

    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Actualiza la tabla Q usando la ecuación de Bellman."""
        
        # Calcula Q actual
        qActual = self.q_table[estado, accion]
        # Visitas
        self.q_visitas[estado, accion] += 1
        alpha = 1.0 / (1.0 + self.q_visitas[estado, accion])
        alpha = max(alpha, 0.1) 

        if terminado:
            maxQ = 0
        else:
            # Calcula el máximo del siguiente estado
            maxQ = np.max(self.q_table[siguiente_estado])

        # Actualiza tabla Q
        self.q_table[estado, accion] = (1 - alpha) * qActual + alpha * (recompensa + self.gamma * maxQ)

         # Decrementar epsilon SOLO al finalizar un episodio
        if terminado:
            if self.epsilon > self.epsilon_min:
                self.epsilon *= self.epsilon_decay
                
    def mostrar_q_table(self):
        print(self.epsilon)
        print("Tabla Q:")
        print(self.q_table)
    
    def mostrar_visitas(self):
        print("Visitas Q:")
        print(self.q_visitas)


Código para entrenar y evaluar el agente de Q-Learning implementado.

In [ ]:
# Ejecución del Agente QLearning Determinista
env = gym.make('FrozenLake-v1', map_name="8x8", is_slippery=True)
agente_qlearning = AgenteQLearning(env.action_space, cant_estados=64)
entrenar_agente(agente_qlearning, env, num_episodios=5000)
resultados = evaluar_agente(agente_qlearning, env, num_episodios=100)
imprimir_resultados_evaluacion(resultados, "Agente QLearning")
env.close()

# Mostrar la tabla Q y las visitas
agente_qlearning.mostrar_q_table()
agente_qlearning.mostrar_visitas()

Episodio 500: Recompensa Promedio = 0.018, Longitud Promedio = 8.5
Episodio 1000: Recompensa Promedio = 0.066, Longitud Promedio = 12.9
Episodio 1500: Recompensa Promedio = 0.184, Longitud Promedio = 18.4
Episodio 2000: Recompensa Promedio = 0.248, Longitud Promedio = 23.0
Episodio 2500: Recompensa Promedio = 0.374, Longitud Promedio = 27.4
Episodio 3000: Recompensa Promedio = 0.544, Longitud Promedio = 32.4
Episodio 3500: Recompensa Promedio = 0.490, Longitud Promedio = 35.4
Episodio 4000: Recompensa Promedio = 0.588, Longitud Promedio = 37.1
Episodio 4500: Recompensa Promedio = 0.634, Longitud Promedio = 37.5
Episodio 5000: Recompensa Promedio = 0.676, Longitud Promedio = 36.8

Agente QLearning - Resultados de Evaluación:
Tasa de Victorias: 71.0%
Recompensa Promedio: 0.710
Desviación Estándar: 0.454
Total de Victorias: 71
0.009998671593271896
Tabla Q:
[[0.53267497 0.45921674 0.46937309 0.44518367]
 [0.24387814 0.31998404 0.24037943 0.36681524]
 [0.26954267 0.26363325 0.25007485 0.255

La siguiente celda define el agente DQN a implementar. 

In [11]:

class DQN(nn.Module):
    """Clase auxiliar que implementa una Red Q Profunda con una capa oculta."""
    
    def __init__(self, tamano_entrada, tamano_oculto, tamano_salida):
        super(DQN, self).__init__()
        pass
    
    def forward(self, x):
        pass

class AgenteDQN(Agente):
    """Agente de Red Q Profunda."""
    
    def __init__(self):
        pass
    
    def elegir_accion(self, estado):
        """Elige acción usando política epsilon-greedy."""
        pass
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        pass
    


Código para entrenar y evaluar el agente de DQN 
implementado.


In [13]:
# Ejecución del Agente DQN Determinista
env = gym.make('FrozenLake-v1')
agente_dqn = AgenteDQN(env.action_space)
entrenar_agente(agente_qlearning, env)
resultados = evaluar_agente(agente_qlearning, env, num_episodios=100)
imprimir_resultados_evaluacion(resultados, "Agente DQN")
env.close()

TypeError: AgenteDQN.__init__() takes 1 positional argument but 2 were given